# Model Comparison

All three models — **Logistic Regression**, **SVM (RBF)**, and **Neural Network** — are compared side-by-side on accuracy, precision, recall, and F1-score for the malignant class on the same 114-sample stratified test set.

In a medical context, **recall for the malignant class is the most important metric** — a false negative (missed cancer) has more serious consequences than a false positive.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Load Data

The same preprocessed 80/20 stratified split used by all models is loaded from `data/processedData/`.

In [ ]:
X_train = pd.read_csv("../data/processedData/X_train_scaled.csv", index_col=0).values
Y_train = pd.read_csv("../data/processedData/Y_train.csv", index_col=0).values.ravel()
X_test  = pd.read_csv("../data/processedData/X_test_scaled.csv", index_col=0).values
Y_test  = pd.read_csv("../data/processedData/Y_test.csv", index_col=0).values.ravel()

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples | Features: {X_train.shape[1]}")

## Train All Models

Each model is trained from scratch on the same training split to ensure a fair comparison.

In [ ]:
import torch
from torch import nn

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, Y_train)
lr_pred = lr.predict(X_test)

# --- SVM (RBF) ---
svm = SVC(kernel="rbf")
svm.fit(X_train, Y_train)
svm_pred = svm.predict(X_test)

# --- Neural Network ---
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(30, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 1)
        )
    def forward(self, x):
        return self.linear_relu_stack(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
nn_model = NeuralNetwork().to(device)
X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
Y_tr = torch.tensor(Y_train, dtype=torch.float32).view(-1, 1).to(device)
X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

loss_fn   = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(nn_model(X_tr), Y_tr)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    nn_preds = (torch.sigmoid(nn_model(X_te)) > 0.5).cpu().numpy().ravel().astype(int)

print("All models trained.")

## Comparison Bar Chart

Grouped bar chart comparing all three models across four metrics. Bars are annotated with exact values.

In [ ]:
def metrics(y_test, y_pred):
    return {
        "Accuracy":               accuracy_score(y_test,  y_pred),
        "Precision\n(malignant)": precision_score(y_test, y_pred),
        "Recall\n(malignant)":    recall_score(y_test,    y_pred),
        "F1-score\n(malignant)":  f1_score(y_test,        y_pred),
    }

results = {
    "Logistic\nRegression": metrics(Y_test, lr_pred),
    "SVM\n(RBF)":           metrics(Y_test, svm_pred),
    "Neural\nNetwork":      metrics(Y_test, nn_preds),
}

metric_names = list(next(iter(results.values())).keys())
model_names  = list(results.keys())
x      = np.arange(len(metric_names))
width  = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Logistic Regression vs. SVM vs. Neural Network")
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0.85, 1.05)
ax.legend(title="Model")
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Results Summary Table

In [ ]:
import pandas as pd

summary = {
    name: {
        "Accuracy":           f"{v['Accuracy']:.4f}",
        "Precision (M)":      f"{v['Precision\\n(malignant)']:.4f}",
        "Recall (M)":         f"{v['Recall\\n(malignant)']:.4f}",
        "F1-score (M)":       f"{v['F1-score\\n(malignant)']:.4f}",
    }
    for name, v in results.items()
}

df_summary = pd.DataFrame(summary).T
df_summary.index.name = "Model"
df_summary